In [ ]:
out_dir <- "results/run1"

In [ ]:
library(ggplot2)
result <- jsonlite::read_json(file.path(out_dir, "result.json"))
donors <- readr::read_tsv(file.path(out_dir, "donors.tsv"), show_col_types = FALSE)
de <- readr::read_tsv(file.path(out_dir, "de.tsv"), show_col_types = FALSE)
md <- \(...) IRdisplay::display_markdown(paste0(...))

# Pseudobulk differential expression: disease vs healthy

## Data

In [ ]:
with(result$counts, md(donors, " donors, ", cells, " cells, ", genes, " genes."))
IRdisplay::display_markdown(paste(knitr::kable(donors), collapse = "\n"))

## Procedure

- **Replication unit:** the donor. Condition varies between donors, so cells of one donor are pseudoreplicates.
- **Aggregation:** raw UMI counts summed over all cells of each donor.
- **Model:** edgeR quasi-likelihood negative binomial GLM, `~ condition`, TMM library sizes.
- **Contrast:** disease vs healthy; log2_fc > 0 means higher in disease.
- **Multiple testing:** Benjamini-Hochberg over every gene that passed the filter.

In [ ]:
with(result$counts, md("**Filter:** edgeR `filterByExpr` kept ", genes_tested, " of ", genes, " genes."))

## Results

In [ ]:
md(result$n_significant, " genes at q < ", result$parameters$q_cutoff, ". Top 10 by p-value:")
IRdisplay::display_markdown(paste(knitr::kable(head(de, 10), digits = 3), collapse = "\n"))

In [ ]:
options(repr.plot.width = 4, repr.plot.height = 3)
ggplot(de, aes(log2_fc, -log10(p_value), colour = q_value < result$parameters$q_cutoff)) +
  geom_point() +
  scale_colour_manual(values = c(`FALSE` = "grey60", `TRUE` = "firebrick"), name = "q < 0.05") +
  labs(x = "log2 fold change (disease vs healthy)", y = "-log10 p-value") +
  theme_bw(8)

*Figure 1.* One point per gene tested; red genes pass the BH cutoff.